In [5]:
import torch
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn.functional as F
from PIL import Image



In [10]:
def load_pretrained_resnet50():
    # ResNet50_Weights.DEFAULT = the best weights torchvision ships,
    # trained on the full ImageNet-1k training set (1.28M images, 1000 classes)
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    model.eval()   # IMPORTANT: inference mode -- dropout off, batch-norm frozen
    return model, weights

In [12]:
model, weights = load_pretrained_resnet50()

In [7]:


def predict_image(image_path, model, weights, top_k=5):
    # weights.transforms() = the EXACT resize/crop/normalize this model expects
    preprocess = weights.transforms()

    img = Image.open(image_path).convert("RGB")
    batch = preprocess(img).unsqueeze(0)   # [3,224,224] -> [1,3,224,224]

    with torch.no_grad():                  # inference, not training -- no gradients needed
        logits = model(batch)              # [1, 1000] raw scores
        probs = F.softmax(logits[0], dim=0)

    categories = weights.meta["categories"]   # the real 1000 ImageNet class names
    top_probs, top_idx = torch.topk(probs, top_k)
    return [(categories[i], p.item()) for p, i in zip(top_probs, top_idx)]

In [13]:
predict_image("espresso_photo.webp", model, weights)

[('espresso', 0.4316655397415161),
 ('cup', 0.018457822501659393),
 ('coffee mug', 0.006382286548614502),
 ('acorn squash', 0.0044325790368020535),
 ('espresso maker', 0.0036926267202943563)]

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

def build_partially_finetuned_resnet(num_classes=2):
    # 1. Load the pretrained model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # 2. Freeze the ENTIRE network first
    for param in model.parameters():
        param.requires_grad = False

    # 3. UNFREEZE only the last convolutional block (layer4)
    for param in model.layer4.parameters():
        param.requires_grad = True

    # 4. Replace the final linear head (trainable by default)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

model = build_partially_finetuned_resnet(num_classes=2)

# 5. Hand ONLY the active parameters to the optimizer
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-5}, # Gentle update for conv filters
    {"params": model.fc.parameters(),     "lr": 1e-3}  # Standard update for brand-new head
])
